# 09 — Xác nhận penalty bằng seed mới

> **English version below.**

Notebook 08 đã chạy/review đủ 24 lượt Windows. Cặp (3,12) có tín hiệu trên choice, chưa vượt P=1 trên reference. Đợt này giữ đúng cặp/khởi tạo đã chọn trước khi xem seed mới; kiểm tra mức cải thiện có lặp lại không.

Chạy **choice trước, reference sau**. Mỗi lần chọn một instance: hai cặp × năm seed = 10 lượt; cả hai instance = 20 lượt. Giữ p=1, shots=1024, maxiter=35 và Aer CPU. Mỗi task vẫn đặt nguyên vẹn lên một host; QUBO/Ising/x/y/slack dùng module của notebook 08.

Chỉ số chính: xác suất placement tối ưu. Chỉ số bổ sung: số lượt tìm được tối ưu, khả thi, gap và chi phí. Năng lượng thô giữa hai cặp có thang khác nhau. Best-valid là hậu xử lý; ghi riêng đồng hạng/mode theo quy tắc đầu ra của proposal.

## Bước 1 — Môi trường và module

Chọn Conda nt409-khiem / Python 3.12 trong VS Code. Cell tìm thư mục repo từ vị trí notebook. Mã nguồn và phiên bản runtime được ghi để tiếp tục đợt chạy an toàn.

In [1]:
import gc
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src/independent_campaign.py").is_file()
             and (p / "data/instances").is_dir()), None)
assert ROOT is not None, "Chép đủ file src và mở notebook bên trong repo."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.qubo import build_qubo, energy_parts
from src.ising import build_ising, ising_energy
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources, run_qaoa_experiment,
    save_experiment_report, experiment_environment,
)
from src.penalty_analysis import penalty_diagnostics
from src.independent_campaign import (
    make_design, make_trials, new_campaign, save_campaign, report_row, verify_campaign,
)
assert sys.version_info[:2] == (3, 12), "Chọn kernel Python 3.12 của dự án."
print("Repo:", ROOT)
print("Runtime:", experiment_environment()["runtime"])
print("Nguồn:", experiment_environment()["source_sha256"])

Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Runtime: {'python_version': '3.12.14', 'platform': 'Windows-11-10.0.26200-SP0', 'numpy_version': '2.5.3', 'scipy_version': '1.18.1', 'qiskit_version': '2.5.2', 'qiskit_optimization_version': '0.7.0', 'qiskit_aer_version': '0.17.2'}
Nguồn: {'placement.py': '7cfa397beb22fdf3feddedc0d889e104a619d6ef3e641ee72a62d82b087deb35', 'qubo.py': '7ac7d64500ce1fe73b1b0a34402d51eb88be33372c3906db4b7197552371e9ae', 'ising.py': '7474b47280084059b259b9fadf2971ef26e97e87a53cff14b9b605a2ee2f8b4b', 'qaoa_experiment.py': 'c02713045e03fb80e1185306e9df527359425033a61f462a10a354f30c674f0c', 'penalty_analysis.py': '1ee33001b1ac723df6d2c9bb541ecac27221e30fa77c1dc9fb9777d3d87363ba', 'independent_campaign.py': 'af953f6f78b4fac9ff49522f3cb8f9677499eb3048401219bd7d411f7ac0c1eb'}


## Bước 2 — Chọn một trong hai đợt xác nhận

Lần đầu giữ STUDY_INSTANCE="p1_choice". Sau khi chạy đủ 10 lượt, lưu notebook có output thành bản choice. Restart Kernel, đổi thành "p0_reference", giữ RESUME_MANIFEST=None và chạy lại các bước; lưu bản có output reference.

Choice: (3,3) đối chứng với (3,12), cùng khởi tạo [0.3,0.05]. Reference: (1,1) đối chứng với (3,12), cùng khởi tạo [0.3,0.7]. Seed 45–49 chưa dùng để chọn ứng viên. Hai hệ số theo thứ tự (assignment, capacity). Không điều chỉnh cặp/khởi tạo sau khi đã xem kết quả seed mới trong cùng protocol.

In [2]:
STUDY_INSTANCE = "p1_choice"  # Sau đợt choice: đổi thành "p0_reference".
PROTOCOLS = {
    "p1_choice": {"control_id": "control_P3", "control_pair": [3, 3],
                  "start_id": "gamma_nho", "initial_point": [0.3, 0.05]},
    "p0_reference": {"control_id": "control_P1", "control_pair": [1, 1],
                     "start_id": "goc_cu", "initial_point": [0.3, 0.7]},
}
assert STUDY_INSTANCE in PROTOCOLS
protocol = PROTOCOLS[STUDY_INSTANCE]
PENALTY_PAIRS = {protocol["control_id"]: protocol["control_pair"], "capacity_12": [3, 12]}
INITIAL_POINTS = {protocol["start_id"]: protocol["initial_point"]}
SEEDS = [45, 46, 47, 48, 49]
BASE_CONFIG = {
    "shots": 1024, "reps": 1, "maxiter": 35,
    "max_qubits": 23, "max_seconds": 300,
    "aer_threads": 4, "aer_memory_mb": 2048, "min_available_gib": 2.0,
}
RESUME_MANIFEST = None
# Khi tiếp tục, đặt đường dẫn confirmation_<instance>_<campaign_id>.json đã in ở Bước 5.
instances = {STUDY_INSTANCE: json.loads(
    (ROOT / "data/instances" / f"{STUDY_INSTANCE}.json").read_text(encoding="utf-8"))}
DESIGN = make_design(instances, PENALTY_PAIRS, INITIAL_POINTS, SEEDS, BASE_CONFIG)
DESIGN["study"] = {
    "stage": "fresh_seed_confirmation", "protocol_version": 1,
    "selection_pilot_campaign_id": "20261006T080102274154Z", "selection_seed": 42,
    "primary_metric": "optimal_placement_probability",
    "candidate_pair_id": "capacity_12", "control_pair_id": protocol["control_id"],
    "fixed_initial_point_id": protocol["start_id"],
    "output_rule": "measured_best_valid_with_frequency_ties_disclosed",
}
planned = make_trials(DESIGN)
assert len(planned) == 10 and 42 not in SEEDS
bundles = {(name, pair_id): build_qubo(item, assignment_penalty=weights[0], capacity_penalty=weights[1])
           for name, item in instances.items() for pair_id, weights in DESIGN["penalty_pairs"].items()}
references = {key: prepare_reference(bundle) for key, bundle in bundles.items()}
print("Đợt:", STUDY_INSTANCE, "| giai đoạn:", DESIGN["study"]["stage"])
print("Khởi tạo:", protocol["initial_point"], "| seed:", SEEDS, "| số lượt:", len(planned))
display(pd.DataFrame([{k: t[k] for k in ["instance_id", "pair_id", "initial_point_id", "seed"]}
                      for t in planned]))

Đợt: p1_choice | giai đoạn: fresh_seed_confirmation
Khởi tạo: [0.3, 0.05] | seed: [45, 46, 47, 48, 49] | số lượt: 10


,instance_id,pair_id,initial_point_id,seed
0,p1_choice,control_P3,gamma_nho,45
1,p1_choice,control_P3,gamma_nho,46
2,p1_choice,control_P3,gamma_nho,47
3,p1_choice,control_P3,gamma_nho,48
4,p1_choice,control_P3,gamma_nho,49
5,p1_choice,capacity_12,gamma_nho,45
6,p1_choice,capacity_12,gamma_nho,46
7,p1_choice,capacity_12,gamma_nho,47
8,p1_choice,capacity_12,gamma_nho,48
9,p1_choice,capacity_12,gamma_nho,49


## Bước 3 — Kiểm tra hai mô hình

Cell dùng cùng kiểm tra đại số/tham chiếu như notebook 08: 20 vector cho hai mô hình. Cặp đối chứng phải có hệ số QUBO đúng bản P chung. Choice: tối ưu gốc 1 host, 20 qubit. Reference: tối ưu gốc 2 host, 23 qubit. PASS xác minh mô hình, chưa đánh giá chất lượng QAOA.

In [3]:
rng = np.random.default_rng(409)
model_rows = []
checked = 0
for (name, pair_id), bundle in bundles.items():
    ref = references[name, pair_id]
    operator, offset = build_ising(bundle)
    vectors = [[0] * operator.num_qubits, ref["example_minimizer_bits"],
               *rng.integers(0, 2, size=(8, operator.num_qubits)).tolist()]
    for bits in vectors:
        formula = energy_parts(bundle, bits)["energy"]
        assert np.isclose(formula, bundle["qubo"].objective.evaluate(bits), rtol=0, atol=1e-7)
        assert np.isclose(formula, ising_energy(operator, offset, bits), rtol=0, atol=1e-7)
        checked += 1
    if bundle["penalty"] is not None:
        old = build_qubo(instances[name], penalty=bundle["penalty"])
        assert old["qubo"].objective.constant == bundle["qubo"].objective.constant
        np.testing.assert_array_equal(old["qubo"].objective.linear.to_array(), bundle["qubo"].objective.linear.to_array())
        np.testing.assert_array_equal(old["qubo"].objective.quadratic.to_array(), bundle["qubo"].objective.quadratic.to_array())
    if ref["instance_feasible"]:
        assert ref["all_qubo_minimizers_feasible"], "Cặp hệ số này có cực tiểu QUBO vi phạm; cần review trước."
    model_rows.append({"instance": name, "pair": pair_id,
        "lambda_assign": bundle["penalties"]["assignment"], "lambda_capacity": bundle["penalties"]["capacity"],
        "qubit": operator.num_qubits, "statevector_MiB": 16 * 2**operator.num_qubits / 2**20,
        "original_optimum": ref["original_best_objective"], "qubo_minimum": ref["minimum_energy"],
        "all_minima_feasible": ref["all_qubo_minimizers_feasible"],
        "uniform_feasible_pct": 100*ref["uniform_baseline"]["feasible_probability"]})
print("PASS — công thức/QUBO/Ising và đối chứng P chung; số vector:", checked)
display(pd.DataFrame(model_rows))

PASS — công thức/QUBO/Ising và đối chứng P chung; số vector: 20


,instance,pair,lambda_assign,lambda_capacity,qubit,statevector_MiB,original_optimum,qubo_minimum,all_minima_feasible,uniform_feasible_pct
0,p1_choice,control_P3,3.0,3.0,20,16.0,1,1.0,True,1.171875
1,p1_choice,capacity_12,3.0,12.0,20,16.0,1,1.0,True,1.171875


## Bước 4 — Đo thử tài nguyên

Mỗi mô hình lấy mẫu một lần trước khi tối ưu. Ước tính thời gian chỉ gồm phần lấy mẫu, không phải cam kết deadline. Aer dùng một statevector tại một thời điểm; max_memory_mb giới hạn bộ nhớ trạng thái lượng tử, không giới hạn toàn bộ Python. RSS đỉnh gồm hoạt động kernel trước đó.

In [4]:
probes = {}
for key, bundle in bundles.items():
    trial = next(t for t in planned if (t["instance_id"], t["pair_id"]) == key)
    print("Đo thử:", *key, flush=True)
    probes[key] = probe_qaoa_resources(bundle, trial["config"])
display(pd.DataFrame([{"instance": key[0], "pair": key[1],
    "sampling_seconds": value["sampling_seconds"],
    "rough_sampling_seconds_per_run": value["rough_maxiter_plus_final_sampling_seconds"],
    "process_peak_rss_mib": value["resources"]["process_peak_rss_mib"]}
    for key, value in probes.items()]))
rough = sum(probes[t["instance_id"], t["pair_id"]]["rough_maxiter_plus_final_sampling_seconds"] for t in planned)
print("Ước tính riêng phần lấy mẫu cả ma trận (phút):", round(rough / 60, 2))
if any(v["rough_maxiter_plus_final_sampling_seconds"] >= BASE_CONFIG["max_seconds"] for v in probes.values()):
    raise RuntimeError("Đo thử chạm ngân sách. Xem tài nguyên trước khi chạy Bước 6.")

Đo thử: p1_choice control_P3
Đo thử: p1_choice capacity_12


,instance,pair,sampling_seconds,rough_sampling_seconds_per_run,process_peak_rss_mib
0,p1_choice,control_P3,0.073521,2.646760,187.527344
1,p1_choice,capacity_12,0.052184,1.878638,188.972656


Ước tính riêng phần lấy mẫu cả ma trận (phút): 0.38


## Bước 5 — Manifest cho đợt xác nhận

Tên file bắt đầu bằng confirmation và chứa tên instance. Thiết kế ghi study.stage=fresh_seed_confirmation cùng mốc pilot đã chọn ứng viên. Dùng đúng manifest instance đang chạy khi resume; notebook xác minh thiết kế/input/mã/runtime và bỏ qua lượt đã hoàn tất.

Các báo cáo vẫn lưu trong phase4b_independent_runs để dùng chung các hàm đã kiểm chứng. Phân biệt pilot và xác nhận bằng metadata study cùng manifest, không gộp hai loại khi tính trung bình/std.

In [5]:
if RESUME_MANIFEST is None:
    campaign = new_campaign(DESIGN)
    manifest_path = ROOT / "data/reference_results/phase4b_independent_runs" / f"confirmation_{STUDY_INSTANCE}_{campaign['campaign_id']}.json"
    save_campaign(manifest_path, campaign)
else:
    manifest_path = (ROOT / RESUME_MANIFEST).resolve()
    assert manifest_path.is_relative_to(ROOT), "Manifest phải nằm trong repo."
    campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    verify_campaign(ROOT, campaign, DESIGN, bundles, references)
print("Manifest:", manifest_path.relative_to(ROOT).as_posix())
print("Giai đoạn:", campaign["design"]["study"]["stage"])
print("Đã hoàn tất / dự kiến:", campaign["completed_run_count"], "/", campaign["planned_run_count"])

Manifest: data/reference_results/phase4b_independent_runs/confirmation_p1_choice_20261006T092857554634Z.json
Giai đoạn: fresh_seed_confirmation
Đã hoàn tất / dự kiến: 0 / 10


## Bước 6 — Chạy mười lượt xác nhận và lưu từng lượt

Hai cặp, năm seed; mỗi seed chạy lại optimizer và lấy mẫu. Lượt hoàn tất có 0 nghiệm vẫn được lưu. Nếu bị ngắt hoặc chạm ngân sách, dừng đợt và giữ các lượt đã hoàn tất; lượt chưa hoàn tất không được ghi thành kết quả. Manifest cập nhật nguyên tử sau mỗi lượt để hỗ trợ resume.

In [6]:
assert STUDY_INSTANCE == DESIGN["instance_ids"][0], "Đổi instance: Restart Kernel và chạy lại từ Bước 1."
completed = {row["run_key"] for row in campaign["completed_runs"]}
for trial in planned:
    if trial["run_key"] in completed:
        print("Bỏ qua lượt đã hoàn tất:", trial["instance_id"], trial["pair_id"], trial["initial_point_id"], trial["seed"])
        continue
    assert experiment_environment() == DESIGN["environment"], "Mã/runtime thay đổi. Tạo đợt mới sau Restart Kernel."
    key = trial["instance_id"], trial["pair_id"]
    print("START", *key, trial["initial_point_id"], "seed", trial["seed"], flush=True)
    report = run_qaoa_experiment(bundles[key], trial["config"], references[key])
    if report["status"] != "completed":
        print("Dừng đợt:", report["stop_reason"], "Lượt chưa hoàn tất không được lưu.")
        del report
        gc.collect()
        break
    assert report["environment"] == DESIGN["environment"]
    report["experiment"] = {"campaign_id": campaign["campaign_id"], "design_hash": campaign["design_hash"],
        "run_key": trial["run_key"], "pair_id": trial["pair_id"],
        "initial_point_id": trial["initial_point_id"], "purpose": "controlled_independent_penalty_pilot"}
    report["study"] = DESIGN["study"].copy()
    started = perf_counter()
    report["penalty_analysis"] = penalty_diagnostics(bundles[key], report)
    report["penalty_analysis_seconds"] = perf_counter() - started
    path = save_experiment_report(ROOT, report)
    row = report_row(ROOT, path, report, trial)
    campaign["completed_runs"].append(row)
    save_campaign(manifest_path, campaign)
    completed.add(trial["run_key"])
    print("DONE | hợp lệ:", round(100*row["feasible_probability"], 4), "% | host tốt nhất:",
          row["observed_best_objective"], "| thời gian:", round(row["elapsed_seconds"], 2), "s", flush=True)
    del report
    gc.collect()

START p1_choice control_P3 gamma_nho seed 45
  evaluation 1: mean E=218.308594, elapsed=0.1s
  evaluation 5: mean E=167.619141, elapsed=0.4s
  evaluation 10: mean E=170.932617, elapsed=0.7s
  evaluation 15: mean E=157.183594, elapsed=1.0s
  evaluation 20: mean E=117.858398, elapsed=1.4s
  evaluation 25: mean E=125.412109, elapsed=1.7s
  evaluation 30: mean E=118.497070, elapsed=2.1s
DONE | hợp lệ: 1.9531 % | host tốt nhất: 1 | thời gian: 2.25 s
START p1_choice control_P3 gamma_nho seed 46
  evaluation 1: mean E=213.098633, elapsed=0.1s
  evaluation 5: mean E=166.795898, elapsed=0.4s
  evaluation 10: mean E=171.289062, elapsed=0.7s
  evaluation 15: mean E=131.041992, elapsed=1.1s
  evaluation 20: mean E=134.976562, elapsed=1.4s
  evaluation 25: mean E=126.777344, elapsed=1.7s
DONE | hợp lệ: 1.3672 % | host tốt nhất: 2 | thời gian: 2.03 s
START p1_choice control_P3 gamma_nho seed 47
  evaluation 1: mean E=216.026367, elapsed=0.1s
  evaluation 5: mean E=193.183594, elapsed=0.4s
  evaluati

## Bước 7 — Mỗi seed là một lượt tối ưu riêng

Mỗi hàng có cùng input, khởi tạo, shots và maxiter; chỉ cặp penalty/seed thay đổi theo thiết kế. Ba cột probability trong bảng đã đổi sang phần trăm. Không tìm thấy nghiệm thì objective/gap là None/NaN; giữ lượt đó trong mẫu thống kê. Std giữa seed sẽ tính ở Bước 8, khác pilot chỉ một seed.

In [7]:
results = pd.DataFrame(campaign["completed_runs"])
if results.empty:
    print("Chưa có lượt hoàn tất.")
else:
    table = results[["instance_id", "pair_id", "initial_point_id", "seed", "assignment_penalty", "capacity_penalty",
        "one_hot_probability", "feasible_probability", "optimal_placement_probability",
        "observed_best_objective", "optimality_gap", "dominant_bitstring_count", "unique_mode_valid", "elapsed_seconds"]].copy()
    for key in ["one_hot_probability", "feasible_probability", "optimal_placement_probability"]:
        table[key] = 100 * pd.to_numeric(table[key])
    display(table.round(6))
    print("Ba cột probability ở bảng này đã đổi sang phần trăm.")

,instance_id,pair_id,initial_point_id,seed,assignment_penalty,capacity_penalty,one_hot_probability,feasible_probability,optimal_placement_probability,observed_best_objective,optimality_gap,dominant_bitstring_count,unique_mode_valid,elapsed_seconds
0,p1_choice,control_P3,gamma_nho,45,3.0,3.0,4.980469,1.953125,0.195312,1,0,3,None,2.251715
1,p1_choice,control_P3,gamma_nho,46,3.0,3.0,4.785156,1.367188,0.000000,2,1,2,None,2.025804
2,p1_choice,control_P3,gamma_nho,47,3.0,3.0,4.785156,1.660156,0.097656,1,0,2,None,1.891785
3,p1_choice,control_P3,gamma_nho,48,3.0,3.0,5.761719,2.050781,0.195312,1,0,1,False,2.085020
4,p1_choice,control_P3,gamma_nho,49,3.0,3.0,6.054688,2.734375,0.097656,1,0,1,False,1.857991
5,p1_choice,capacity_12,gamma_nho,45,3.0,12.0,6.640625,2.050781,0.292969,1,0,1,False,1.783781
6,p1_choice,capacity_12,gamma_nho,46,3.0,12.0,5.566406,1.855469,0.195312,1,0,3,None,1.969171
7,p1_choice,capacity_12,gamma_nho,47,3.0,12.0,7.226562,0.878906,0.097656,1,0,1,False,1.912806
8,p1_choice,capacity_12,gamma_nho,48,3.0,12.0,6.347656,1.367188,0.000000,2,1,1024,None,1.850054
9,p1_choice,capacity_12,gamma_nho,49,3.0,12.0,6.445312,1.660156,0.390625,1,0,4,None,2.026672


Ba cột probability ở bảng này đã đổi sang phần trăm.


## Bước 8 — Tổng hợp và so sánh theo từng seed

Mỗi cặp phải có năm lượt hoàn tất. Bảng đầu ghi trung bình và độ lệch chuẩn giữa seed, số lượt có nghiệm/tối ưu và tổng mẫu tối ưu. Bảng sau ghép đúng seed để xem chênh lệch ứng viên−đối chứng theo điểm phần trăm.

Review theo thứ tự: xác suất placement tối ưu, số lượt tối ưu, mức nhất quán giữa seed, khả thi và chi phí. Chênh lệch dương ở một seed chưa chứng minh cải thiện ổn định; năm seed là xác nhận có giới hạn trên fixture hiện tại. Không tự điều chỉnh cấu hình để làm đẹp bảng hoặc tuyên bố ưu thế lượng tử.

In [8]:
summary = pd.DataFrame()
paired = pd.DataFrame()
if not results.empty:
    shots = BASE_CONFIG["shots"]
    summary = results.groupby("pair_id", sort=False).agg(
        completed_runs=("run_key", "size"), valid_runs=("found_valid", "sum"),
        optimal_runs=("found_optimal", "sum"),
        optimal_shots=("optimal_placement_probability", lambda s: int(round(shots*s.sum()))),
        optimal_pct_mean=("optimal_placement_probability", lambda s: 100*s.mean()),
        optimal_pct_std=("optimal_placement_probability", lambda s: 100*s.std(ddof=1)),
        feasible_pct_mean=("feasible_probability", lambda s: 100*s.mean()),
        feasible_pct_std=("feasible_probability", lambda s: 100*s.std(ddof=1)),
        onehot_pct_mean=("one_hot_probability", lambda s: 100*s.mean()),
        mean_solver_seconds=("elapsed_seconds", "mean"),
        total_solver_seconds=("elapsed_seconds", "sum"),
        max_recorded_peak_rss_mib=("process_peak_rss_mib", "max"),
    ).reset_index()
    display(summary.round(6))
    control = results[results["pair_id"] == protocol["control_id"]].set_index("seed")
    candidate = results[results["pair_id"] == "capacity_12"].set_index("seed")
    common_seeds = sorted(set(control.index) & set(candidate.index))
    paired = pd.DataFrame([{
        "seed": seed,
        "control_optimal_shots": int(round(shots*control.loc[seed, "optimal_placement_probability"])),
        "candidate_optimal_shots": int(round(shots*candidate.loc[seed, "optimal_placement_probability"])),
        "delta_optimal_pct_points": 100*(candidate.loc[seed, "optimal_placement_probability"]-control.loc[seed, "optimal_placement_probability"]),
        "delta_feasible_pct_points": 100*(candidate.loc[seed, "feasible_probability"]-control.loc[seed, "feasible_probability"]),
        "control_found_optimal": bool(control.loc[seed, "found_optimal"]),
        "candidate_found_optimal": bool(candidate.loc[seed, "found_optimal"]),
    } for seed in common_seeds])
    display(paired.round(6))
    if not paired.empty:
        delta = paired["delta_optimal_pct_points"]
        print("Seed có nhiều mẫu tối ưu hơn / bằng / ít hơn đối chứng:",
              int((delta>0).sum()), "/", int((delta==0).sum()), "/", int((delta<0).sum()))
    print("Đã ghép:", len(paired), "/", len(SEEDS), "seed. Cần đủ cả hai cặp trước khi kết luận.")
    print("Baseline placement tối ưu đều (%):", 100*next(iter(references.values()))["uniform_baseline"]["optimal_placement_probability"])
    print("Std là độ lệch chuẩn giữa các lần chạy seed; không phải khoảng tin cậy.")

,pair_id,completed_runs,valid_runs,optimal_runs,optimal_shots,optimal_pct_mean,optimal_pct_std,feasible_pct_mean,feasible_pct_std,onehot_pct_mean,mean_solver_seconds,total_solver_seconds,max_recorded_peak_rss_mib
0,control_P3,5,5,4,6,0.117188,0.081705,1.953125,0.512114,5.273438,2.022463,10.112315,195.808594
1,capacity_12,5,5,4,10,0.195312,0.154408,1.562500,0.458048,6.445312,1.908497,9.542484,195.898438


,seed,control_optimal_shots,candidate_optimal_shots,delta_optimal_pct_points,delta_feasible_pct_points,control_found_optimal,candidate_found_optimal
0,45,2,3,0.097656,0.097656,True,True
1,46,0,2,0.195312,0.488281,False,True
2,47,1,1,0.000000,-0.781250,True,True
3,48,2,0,-0.195312,-0.683594,True,False
4,49,1,4,0.292969,-1.074219,True,True


Seed có nhiều mẫu tối ưu hơn / bằng / ít hơn đối chứng: 3 / 1 / 1
Đã ghép: 5 / 5 seed. Cần đủ cả hai cặp trước khi kết luận.
Baseline placement tối ưu đều (%): 0.09765625
Std là độ lệch chuẩn giữa các lần chạy seed; không phải khoảng tin cậy.


## Bước 9 — Xem mẫu đo của một lượt

Tần suất cao nhất và nghiệm hợp lệ tốt nhất là hai quy tắc đầu ra khác nhau. Chương trình không sửa task/slack trước khi chấm. Chọn INSPECT_RUN_INDEX để xem một lượt khác.

In [9]:
INSPECT_RUN_INDEX = 0
if campaign["completed_runs"]:
    entry = campaign["completed_runs"][INSPECT_RUN_INDEX]
    inspected = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
    print("Nguồn:", entry["report"])
    print("Giai đoạn xác nhận:", inspected["study"])
    print("Hai hệ số:", inspected["penalties"], "| thứ tự tham số:", inspected["parameter_names"])
    print("Nghiệm hợp lệ tốt nhất:", inspected["metrics"]["best_observed_valid"])
    print("Chẩn đoán:", inspected["penalty_analysis"])
    display(pd.DataFrame([{"probability": s["probability"], "energy": s["energy"],
        "feasible": s["decoded"]["feasible"], "hosts": s["decoded"]["objective"],
        "x": s["decoded"]["x"], "y": s["decoded"]["y"]} for s in inspected["final_samples"][:5]]))
    del inspected
    gc.collect()

Nguồn: data/reference_results/phase4b_independent_runs/p1_choice_a3_c3_p1_seed45_20261006T092953905370Z.json
Giai đoạn xác nhận: {'stage': 'fresh_seed_confirmation', 'protocol_version': 1, 'selection_pilot_campaign_id': '20261006T080102274154Z', 'selection_seed': 42, 'primary_metric': 'optimal_placement_probability', 'candidate_pair_id': 'capacity_12', 'control_pair_id': 'control_P3', 'fixed_initial_point_id': 'gamma_nho', 'output_rule': 'measured_best_valid_with_frequency_ties_disclosed'}
Hai hệ số: {'assignment': 3.0, 'capacity': 3.0} | thứ tự tham số: ['β[0]', 'γ[0]']
Nghiệm hợp lệ tốt nhất: {'bits': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0], 'energy': 58.0, 'probability': 0.0009765625, 'feasible': True, 'objective': 1, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[1, 0], [1, 0], [1, 0], [1, 0]], 'y': [1, 0], 'usage': [[5, 5], [0, 0]]}
Chẩn đoán: {'mean_active_hosts': 1.478515625, 'mean_assignment_error': 2.0380859375, 'mean_capacity_equation_error': 36.24511718

,probability,energy,feasible,hosts,x,y
0,0.001953,73.0,False,None,"[[1, 0], [1, 0], [0, 0], [1, 1]]","[1, 0]"
1,0.001953,110.0,False,None,"[[1, 1], [0, 0], [0, 0], [0, 1]]","[1, 1]"
2,0.001953,47.0,False,None,"[[1, 0], [1, 0], [0, 0], [1, 0]]","[1, 1]"
3,0.000977,14.0,False,None,"[[0, 1], [0, 0], [1, 0], [1, 1]]","[1, 1]"
4,0.000977,139.0,False,None,"[[0, 0], [1, 1], [0, 1], [1, 0]]","[1, 0]"


## Bước 10 — Đủ mười lượt và dữ liệu khớp

PASS kiểm tra manifest/mẫu/metrics/metadata; chất lượng phải đọc ở Bước 7–9. Mỗi instance hoàn tất cần 10/10, không còn lượt, năm lượt ở mỗi cặp và năm seed được ghép. Lưu một bản notebook có output cho instance này. Cả nghiên cứu xác nhận chỉ hoàn tất sau khi choice và reference đều đủ mười lượt.

In [10]:
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
verified = verify_campaign(ROOT, saved, DESIGN, bundles, references)
assert verified == campaign["completed_runs"]
for entry in verified:
    source = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
    assert source["study"] == DESIGN["study"], "Metadata xác nhận không khớp."
    assert source["config"]["seed"] in SEEDS and source["config"]["seed"] != 42
    del source
if saved["completed_run_count"] == len(planned):
    assert summary["completed_runs"].tolist() == [5, 5] and len(paired) == 5
print("PASS — manifest, mẫu đo, metrics và metadata xác nhận khớp.")
print("Instance:", STUDY_INSTANCE, "| hoàn tất / dự kiến:", saved["completed_run_count"], "/", saved["planned_run_count"])
print("Lượt còn lại:", len(saved["remaining_run_keys"]))
print("Gửi: notebook có output, manifest", manifest_path.name, "và các JSON nguồn được tham chiếu.")
print("Thư mục:", manifest_path.parent.relative_to(ROOT).as_posix())

PASS — manifest, mẫu đo, metrics và metadata xác nhận khớp.
Instance: p1_choice | hoàn tất / dự kiến: 10 / 10
Lượt còn lại: 0
Gửi: notebook có output, manifest confirmation_p1_choice_20261006T092857554634Z.json và các JSON nguồn được tham chiếu.
Thư mục: data/reference_results/phase4b_independent_runs


## Sau khi chạy cả hai instance

Giữ riêng kết quả pilot seed 42 và xác nhận seed 45–49. Gửi hai notebook có output, hai manifest confirmation và 20 báo cáo mới. Review xong mới chốt cặp penalty cho độ sâu p=2; nếu không cải thiện rõ, kết thúc nhánh tinh chỉnh penalty và tiếp tục nghiên cứu độ sâu với đối chứng p=1. Không mở thêm ma trận penalty từ kết quả xác nhận để rồi gọi chính các seed đó là xác nhận mới.

Resume: giữ nguyên instance/thiết kế/mã/runtime, Restart Kernel, đặt RESUME_MANIFEST đến đúng file confirmation, chạy các bước theo thứ tự. Bước 6 bỏ qua lượt đã có. Khi chuyển instance, Restart Kernel và đặt RESUME_MANIFEST=None. Lưu output choice trước khi chạy reference để có cả hai bảng.

---

## English explanation

This notebook confirms the single candidate (assignment,capacity)=(3,12) selected from the reviewed 24-trial Windows pilot. Run choice first, then reference in a restarted kernel. Each study has two penalty pairs, one fixed start and fresh seeds 45–49: ten trials per instance, twenty in total. Choice compares against (3,3) at [0.3,0.05]; reference compares against (1,1) at [0.3,0.7]. Keep depth 1, shots 1024, COBYLA maxiter 35 and the existing CPU Aer limits. Initial angles follow beta[0],gamma[0].

Reuse notebook 08 modules unchanged. Run steps 1–10: environment, fixed design, two-model verification, resource probes, manifest creation/resume, sequential runs, per-seed results, grouped and seed-paired comparisons, raw sample inspection and final consistency/completeness checks. Expect ten completed trials, five per pair and five paired seeds for each instance. Save separate executed choice/reference notebook copies.

Reports continue to use schema 3 and the existing phase4b storage/helper scope for compatibility. Confirmation identity is explicit in design.study and report.study with stage=fresh_seed_confirmation, protocol version, selection pilot campaign/seed and the primary metric; design hashes and final checks protect this metadata. Manifests are named confirmation_<instance>_<campaign_id>.json. Keep pilot and confirmation separate in statistics. Resume requires identical design/input/code/runtime and cannot cross instances.

Primary quality metric: mean optimal-placement probability across seeds. Also inspect successful trials, standard deviation, paired differences, feasibility, gaps and solver/memory costs. Standard deviation is not a confidence interval; five seeds on existing fixtures do not establish generalization or quantum advantage. Use only measured bits, retain completed zero-feasibility trials and disclose best-valid postselection and frequency ties. Do not rank raw energies across weight scales or retune after examining confirmation outcomes without defining a new exploratory study.

Submit two executed notebooks, two confirmation manifests and twenty referenced report JSONs. After review, choose the weights for a resource-probed depth-2 study with a depth-1 control; when optimizer budgets increase, retain a matched depth-1 budget control. Software verification uses reduced-budget Linux runs and does not predict Windows experimental quality.